# 렉스턴 스포츠 외관 부품 13개 클래스 학습

먼저 Colab 메뉴에서 **런타임 → 런타임 유형 변경 → GPU**를 선택하세요. 무료 GPU는 제공 여부와 사용 시간이 보장되지 않습니다. 현재 데이터는 17장(학습 15장, 검증 2장)이므로 이 실행은 학습 파이프라인 확인용이며 정확도 평가용이 아닙니다.

준비물: 프로젝트의 `dataset_13class.zip` 파일

In [ ]:
%pip -q install "ultralytics>=8.3,<9"

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()  # dataset_13class.zip을 선택
zip_name = next(iter(uploaded))
work = Path('/content/rexton_training')
work.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(work)
yolo_dir = work / 'dataset' / 'yolo'
data_yaml = yolo_dir / 'data.yaml'
assert data_yaml.is_file(), f'학습 설정 파일을 찾을 수 없습니다: {data_yaml}'
# ZIP 안의 상대경로를 현재 Colab 위치에 맞춰 절대경로로 바꿉니다.
yaml_text = data_yaml.read_text(encoding='utf-8').replace('path: .', f'path: {yolo_dir}')
data_yaml.write_text(yaml_text, encoding='utf-8')
print('데이터 설정:', data_yaml)

In [ ]:
import torch
print('CUDA 사용 가능:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU 런타임이 아닙니다. 런타임 유형에서 GPU를 선택하고 다시 실행하세요.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
results = model.train(
    data=str(data_yaml),
    epochs=100,
    patience=15,
    imgsz=640,
    batch=4,
    device=0,
    workers=2,
    project=str(work / 'runs'),
    name='rexton_sports_13class',
    exist_ok=True,
    seed=42
)
best_pt = work / 'runs' / 'rexton_sports_13class' / 'weights' / 'best.pt'
assert best_pt.is_file(), f'학습 가중치를 찾을 수 없습니다: {best_pt}'
print('최적 가중치:', best_pt)

In [ ]:
# 실행이 끝나면 best.pt를 PC로 다운로드합니다.
files.download(str(best_pt))

## PC 프로그램에서 사용하기

다운로드한 `best.pt`를 `repo_upload/models/rexton_sports/best.pt`로 복사하세요. `models/rexton_sports` 폴더가 없다면 만드세요. 이후 13개 클래스와 이름 순서가 일치하는지 확인한 뒤 프로그램에서 탐지를 실행합니다. Colab 세션이 끝나면 런타임 파일은 사라질 수 있으므로 결과 가중치를 반드시 다운로드하세요.